# PyroFinder — YOLO11n Baseline Training on Kaggle

This notebook trains **YOLO11n** on **D-Fire** for two object-detection classes:

- `0 = smoke`
- `1 = fire`

It is adapted for **Kaggle Notebooks**.

Expected Kaggle setup:

1. Create a private Kaggle Dataset containing `D-Fire.zip`.
2. Create a Kaggle Notebook and attach that dataset via **Add Input**.
3. In Notebook settings, enable **Accelerator → GPU** and **Internet → On**.
4. Run all cells.

Outputs are written to:

`/kaggle/working/pyrofinder_results/`

Final files:

- `baseline_yolo11n.json`
- `yolo11n_dfire_best.pt`
- `pyrofinder_yolo11n_results.zip`

## Step 1 — Check Kaggle GPU

In [ ]:
from pathlib import Path
import os, subprocess, sys

print("Python:", sys.version)
print("Kaggle input exists:", Path("/kaggle/input").exists())
print("Kaggle working exists:", Path("/kaggle/working").exists())

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    else:
        raise RuntimeError("No GPU detected. In Kaggle Notebook Settings, set Accelerator → GPU, then restart/run again.")
except Exception as exc:
    print("GPU check failed:", exc)
    raise

## Step 2 — Install / verify dependencies

In [ ]:
# Kaggle usually has many ML packages preinstalled, but Ultralytics may need install/update.
# Internet must be ON in Kaggle notebook settings for this cell.
!pip install -q "ultralytics>=8.3" PyYAML

import ultralytics
print("Ultralytics:", ultralytics.__version__)

## Step 3 — Configuration

In [ ]:
from pathlib import Path

# You normally do NOT need to edit this cell.
# The notebook will search /kaggle/input for D-Fire.zip or a D-Fire folder.

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")

# Where the dataset will be unzipped/copied for fast local training.
DFIRE_UNZIP_DIR = WORKING_ROOT / "D-Fire"

# All model outputs/checkpoints go here.
RESULTS_DIR = WORKING_ROOT / "pyrofinder_results"
RUNS_PROJECT = RESULTS_DIR / "runs"
RUN_NAME = "yolo11n_dfire_baseline"
RUN_DIR = RUNS_PROJECT / RUN_NAME

# Training settings.
IMGSZ = 640
EPOCHS = 30
BATCH = 16
RESUME_IF_AVAILABLE = True

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_PROJECT.mkdir(parents=True, exist_ok=True)

print("Input root:", KAGGLE_INPUT_ROOT)
print("Results dir:", RESULTS_DIR)
print("YOLO run dir:", RUN_DIR)

## Step 4 — Find and load D-Fire

In [ ]:
from pathlib import Path
import zipfile, shutil

def looks_like_dfire_root(p: Path) -> bool:
    return (p / "train" / "images").exists() and (p / "train" / "labels").exists() and (p / "test" / "images").exists() and (p / "test" / "labels").exists()

def find_dfire_zip() -> Path | None:
    zips = []
    for pattern in ["D-Fire.zip", "DFire.zip", "*D*Fire*.zip", "*dfire*.zip", "*.zip"]:
        zips.extend(KAGGLE_INPUT_ROOT.rglob(pattern))
    zips = sorted(set(zips), key=lambda p: ("d-fire" not in p.name.lower() and "dfire" not in p.name.lower(), len(str(p))))
    return zips[0] if zips else None

def find_dfire_folder() -> Path | None:
    candidates = []
    for p in KAGGLE_INPUT_ROOT.rglob("*"):
        if p.is_dir() and looks_like_dfire_root(p):
            candidates.append(p)
    return sorted(candidates, key=lambda p: len(str(p)))[0] if candidates else None

folder = find_dfire_folder()
zip_path = find_dfire_zip()

if folder:
    DFIRE_ROOT = folder
    print("Found D-Fire folder in Kaggle input:", DFIRE_ROOT)
elif zip_path:
    print("Found D-Fire zip:", zip_path)
    if not DFIRE_UNZIP_DIR.exists():
        print("Unzipping to:", DFIRE_UNZIP_DIR)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(DFIRE_UNZIP_DIR)
        print("Unzip complete.")
    else:
        print("Already unzipped:", DFIRE_UNZIP_DIR)

    # Handle either /kaggle/working/D-Fire/train/... or /kaggle/working/D-Fire/D-Fire/train/...
    if looks_like_dfire_root(DFIRE_UNZIP_DIR):
        DFIRE_ROOT = DFIRE_UNZIP_DIR
    else:
        nested = [p for p in DFIRE_UNZIP_DIR.rglob("*") if p.is_dir() and looks_like_dfire_root(p)]
        if not nested:
            raise FileNotFoundError("D-Fire.zip was found and unzipped, but train/test images+labels folders were not found.")
        DFIRE_ROOT = sorted(nested, key=lambda p: len(str(p)))[0]
    print("D-Fire root:", DFIRE_ROOT)
else:
    raise FileNotFoundError(
        "Could not find D-Fire.zip or D-Fire folder under /kaggle/input. "
        "Create a Kaggle Dataset with D-Fire.zip and attach it to this notebook using Add Input."
    )

# Quick dataset sanity check
for rel in ["train/images", "train/labels", "test/images", "test/labels"]:
    p = DFIRE_ROOT / rel
    print(rel, "exists=", p.exists(), "count=", len(list(p.glob('*'))) if p.exists() else 0)

## Step 5 — Create YOLO data.yaml

In [ ]:
import yaml
from pathlib import Path

yaml_path = Path("/kaggle/working/dfire_yolo11n.yaml")

data = {
    "path": str(DFIRE_ROOT),
    "train": "train/images",
    "val": "test/images",
    "nc": 2,
    "names": {0: "smoke", 1: "fire"},
}

with yaml_path.open("w", encoding="utf-8") as f:
    yaml.dump(data, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print("YAML written:", yaml_path)
print(yaml_path.read_text())

## Step 6 — Train YOLO11n and export artifacts

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PyroFinder YOLO11n — Kaggle train + export
# ─────────────────────────────────────────────────────────────────────────────

from pathlib import Path
import json, shutil, zipfile
from datetime import date
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

RESULTS_DIR = Path(globals().get("RESULTS_DIR", "/kaggle/working/pyrofinder_results"))
RUNS_PROJECT = Path(globals().get("RUNS_PROJECT", RESULTS_DIR / "runs"))
RUN_NAME = globals().get("RUN_NAME", "yolo11n_dfire_baseline")
RUN_DIR = RUNS_PROJECT / RUN_NAME
WEIGHTS_DIR = RUN_DIR / "weights"
yaml_path = Path(globals().get("yaml_path", "/kaggle/working/dfire_yolo11n.yaml"))
IMGSZ = int(globals().get("IMGSZ", 640))
EPOCHS = int(globals().get("EPOCHS", 30))
BATCH = int(globals().get("BATCH", 16))
RESUME_IF_AVAILABLE = bool(globals().get("RESUME_IF_AVAILABLE", True))
device = "0" if torch.cuda.is_available() else "cpu"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_PROJECT.mkdir(parents=True, exist_ok=True)

print("Device:", device)
print("Results dir:", RESULTS_DIR)
print("YOLO run dir:", RUN_DIR)
print("YAML:", yaml_path)

if not yaml_path.exists():
    raise FileNotFoundError(f"Missing YAML: {yaml_path}. Re-run Step 5.")

def _scalar(x):
    try:
        if isinstance(x, (list, tuple, np.ndarray)):
            x = x[0] if len(x) else None
        return round(float(x), 4) if x is not None else None
    except Exception:
        return None

def _find_metric_col(columns, *needles):
    for col in list(columns):
        low = str(col).lower().replace(" ", "")
        if all(n.lower().replace(" ", "") in low for n in needles):
            return col
    return None

def _metrics_from_results_csv(results_csv: Path) -> tuple[dict, int | None]:
    if not results_csv.exists():
        return {
            "map50": None,
            "map50_95": None,
            "precision": None,
            "recall": None,
            "f1": None,
            "metrics_source": "results.csv not found",
        }, None

    df = pd.read_csv(results_csv)
    df.columns = [str(c).strip() for c in df.columns]

    map50_col = _find_metric_col(df.columns, "map50")
    map_col = _find_metric_col(df.columns, "map50-95") or _find_metric_col(df.columns, "map")
    p_col = _find_metric_col(df.columns, "precision")
    r_col = _find_metric_col(df.columns, "recall")
    epoch_col = _find_metric_col(df.columns, "epoch")

    if map50_col and df[map50_col].notna().any():
        best_idx = df[map50_col].astype(float).idxmax()
    elif map_col and df[map_col].notna().any():
        best_idx = df[map_col].astype(float).idxmax()
    else:
        best_idx = df.index[-1]

    row = df.loc[best_idx]
    precision = _scalar(row[p_col]) if p_col else None
    recall = _scalar(row[r_col]) if r_col else None
    f1 = (
        round(2 * precision * recall / (precision + recall), 4)
        if precision is not None and recall is not None and (precision + recall) > 0
        else None
    )

    selected_epoch = None
    if epoch_col:
        try:
            selected_epoch = int(row[epoch_col])
        except Exception:
            selected_epoch = int(best_idx) + 1
    else:
        selected_epoch = int(best_idx) + 1

    return {
        "map50": _scalar(row[map50_col]) if map50_col else None,
        "map50_95": _scalar(row[map_col]) if map_col else None,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "metrics_source": f"recovered_from_{results_csv}",
        "selected_epoch_or_row": selected_epoch,
    }, selected_epoch

def _latest_checkpoint() -> Path | None:
    candidates = []
    for root in [RUN_DIR, Path("/kaggle/working")]:
        if root.exists():
            candidates.extend(root.rglob("best.pt"))
            candidates.extend(root.rglob("last.pt"))
            candidates.extend(root.rglob("epoch*.pt"))
    candidates = [p for p in candidates if p.exists()]
    if not candidates:
        return None
    bests = [p for p in candidates if p.name == "best.pt"]
    if bests:
        return sorted(bests, key=lambda p: p.stat().st_mtime, reverse=True)[0]
    lasts = [p for p in candidates if p.name == "last.pt"]
    if lasts:
        return sorted(lasts, key=lambda p: p.stat().st_mtime, reverse=True)[0]
    return sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0]

def export_artifacts(status: str):
    ckpt = _latest_checkpoint()
    if ckpt is None:
        raise FileNotFoundError("No YOLO checkpoint found to export.")

    print("Checkpoint selected:", ckpt)
    final_weights = RESULTS_DIR / "yolo11n_dfire_best.pt"
    shutil.copy2(ckpt, final_weights)

    results_csv_candidates = []
    for root in [RUN_DIR, Path("/kaggle/working")]:
        if root.exists():
            results_csv_candidates.extend(root.rglob("results.csv"))
    results_csv = sorted(results_csv_candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0] if results_csv_candidates else RUN_DIR / "results.csv"

    metrics, selected_epoch = _metrics_from_results_csv(results_csv)

    result_doc = {
        "model_name": "YOLO11n (object detection baseline)",
        "model_family": "object_detection",
        "run_date": date.today().isoformat(),
        "status": status,
        "task": "two-class object detection: fire / smoke",
        "dataset": {
            "name": "D-Fire",
            "source": "Kaggle input dataset containing D-Fire.zip or D-Fire folder",
            "train_size": 17221,
            "test_size": 4306,
            "class_mapping": {"0": "smoke", "1": "fire"},
            "label_format": "YOLO normalized bounding boxes",
            "evaluation_split": "test folder used as YOLO val split",
        },
        "model_params": {
            "model": "YOLO11n",
            "starting_weights": "yolo11n.pt",
            "trained_weights": str(final_weights),
            "checkpoint_source": str(ckpt),
            "imgsz": IMGSZ,
            "epochs_requested": EPOCHS,
            "selected_epoch_or_row": selected_epoch,
            "batch": BATCH,
            "device": device,
            "run_dir": str(RUN_DIR),
        },
        "metrics": metrics,
        "notes": (
            "YOLO11n is the lightweight object-detection baseline for PyroFinder. "
            "Metrics are recovered from YOLO results.csv, selecting the best mAP50 row when available."
        ),
    }

    final_json = RESULTS_DIR / "baseline_yolo11n.json"
    final_json.write_text(json.dumps(result_doc, indent=2, ensure_ascii=False), encoding="utf-8")

    zip_path = RESULTS_DIR / "pyrofinder_yolo11n_results.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(final_json, arcname="baseline_yolo11n.json")
        zf.write(final_weights, arcname="yolo11n_dfire_best.pt")
        if results_csv.exists():
            zf.write(results_csv, arcname="results.csv")

    print("\nSaved final artifacts:")
    for p in [final_json, final_weights, zip_path]:
        print("OK " if p.exists() else "MISS ", p)

    print("\nMetrics:")
    print(json.dumps(metrics, indent=2))

# Train or resume.
training_status = "completed"
try:
    last_ckpt = WEIGHTS_DIR / "last.pt"
    if RESUME_IF_AVAILABLE and last_ckpt.exists():
        print("Resuming from checkpoint:", last_ckpt)
        model = YOLO(str(last_ckpt))
        model.train(resume=True)
    else:
        print("Starting new YOLO11n training.")
        model = YOLO("yolo11n.pt")
        model.train(
            data=str(yaml_path),
            epochs=EPOCHS,
            imgsz=IMGSZ,
            batch=BATCH,
            device=device,
            project=str(RUNS_PROJECT),
            name=RUN_NAME,
            exist_ok=True,
            save=True,
            save_period=1,
            verbose=True,
        )
except Exception as exc:
    training_status = f"interrupted_or_failed: {type(exc).__name__}: {exc}"
    print("Training did not finish cleanly:", training_status)
    print("Trying to export the latest checkpoint already saved in /kaggle/working...")
    try:
        export_artifacts(training_status)
    finally:
        raise
else:
    export_artifacts(training_status)

## Step 7 — Recovery / final check only

Run this cell manually if training stopped but the Kaggle session is still alive. It does **not** train.

In [ ]:
from pathlib import Path
import json, shutil, zipfile
from datetime import date
import numpy as np
import pandas as pd

RESULTS_DIR = Path(globals().get("RESULTS_DIR", "/kaggle/working/pyrofinder_results"))
RUNS_PROJECT = Path(globals().get("RUNS_PROJECT", RESULTS_DIR / "runs"))
RUN_NAME = globals().get("RUN_NAME", "yolo11n_dfire_baseline")
RUN_DIR = RUNS_PROJECT / RUN_NAME
IMGSZ = int(globals().get("IMGSZ", 640))
EPOCHS = int(globals().get("EPOCHS", 30))
BATCH = int(globals().get("BATCH", 16))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Searching for checkpoints under:", RUN_DIR)

checkpoint_candidates = []
for root in [RUN_DIR, Path("/kaggle/working")]:
    if root.exists():
        checkpoint_candidates.extend(root.rglob("best.pt"))
        checkpoint_candidates.extend(root.rglob("last.pt"))
        checkpoint_candidates.extend(root.rglob("epoch*.pt"))
checkpoint_candidates = [p for p in checkpoint_candidates if p.exists()]

if not checkpoint_candidates:
    raise FileNotFoundError("No checkpoint found in /kaggle/working. If the notebook session reset, the artifacts may be gone.")

bests = [p for p in checkpoint_candidates if p.name == "best.pt"]
lasts = [p for p in checkpoint_candidates if p.name == "last.pt"]
if bests:
    ckpt = sorted(bests, key=lambda p: p.stat().st_mtime, reverse=True)[0]
elif lasts:
    ckpt = sorted(lasts, key=lambda p: p.stat().st_mtime, reverse=True)[0]
else:
    ckpt = sorted(checkpoint_candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0]

print("Recovered checkpoint:", ckpt)
final_weights = RESULTS_DIR / "yolo11n_dfire_best.pt"
shutil.copy2(ckpt, final_weights)

# Recover simple metrics from results.csv
results_csv_candidates = []
for root in [RUN_DIR, Path("/kaggle/working")]:
    if root.exists():
        results_csv_candidates.extend(root.rglob("results.csv"))
results_csv = sorted(results_csv_candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0] if results_csv_candidates else None

def _find_metric_col(columns, *needles):
    for col in list(columns):
        low = str(col).lower().replace(" ", "")
        if all(n.lower().replace(" ", "") in low for n in needles):
            return col
    return None

metrics = {"map50": None, "map50_95": None, "precision": None, "recall": None, "f1": None, "metrics_source": "results.csv not found"}
selected_epoch = None
if results_csv and results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [str(c).strip() for c in df.columns]
    map50_col = _find_metric_col(df.columns, "map50")
    map_col = _find_metric_col(df.columns, "map50-95") or _find_metric_col(df.columns, "map")
    p_col = _find_metric_col(df.columns, "precision")
    r_col = _find_metric_col(df.columns, "recall")
    epoch_col = _find_metric_col(df.columns, "epoch")
    best_idx = df[map50_col].astype(float).idxmax() if map50_col and df[map50_col].notna().any() else df.index[-1]
    row = df.loc[best_idx]
    precision = round(float(row[p_col]), 4) if p_col else None
    recall = round(float(row[r_col]), 4) if r_col else None
    f1 = round(2 * precision * recall / (precision + recall), 4) if precision and recall and (precision + recall) > 0 else None
    selected_epoch = int(row[epoch_col]) if epoch_col else int(best_idx) + 1
    metrics = {
        "map50": round(float(row[map50_col]), 4) if map50_col else None,
        "map50_95": round(float(row[map_col]), 4) if map_col else None,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "metrics_source": f"recovered_from_{results_csv}",
        "selected_epoch_or_row": selected_epoch,
    }

result_doc = {
    "model_name": "YOLO11n (object detection baseline)",
    "model_family": "object_detection",
    "run_date": date.today().isoformat(),
    "status": "recovered_or_final_check",
    "task": "two-class object detection: fire / smoke",
    "dataset": {"name": "D-Fire", "train_size": 17221, "test_size": 4306, "class_mapping": {"0": "smoke", "1": "fire"}},
    "model_params": {"model": "YOLO11n", "trained_weights": str(final_weights), "checkpoint_source": str(ckpt), "imgsz": IMGSZ, "epochs_requested": EPOCHS, "selected_epoch_or_row": selected_epoch, "batch": BATCH, "run_dir": str(RUN_DIR)},
    "metrics": metrics,
}

final_json = RESULTS_DIR / "baseline_yolo11n.json"
final_json.write_text(json.dumps(result_doc, indent=2, ensure_ascii=False), encoding="utf-8")
zip_path = RESULTS_DIR / "pyrofinder_yolo11n_results.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(final_json, arcname="baseline_yolo11n.json")
    zf.write(final_weights, arcname="yolo11n_dfire_best.pt")
    if results_csv and results_csv.exists():
        zf.write(results_csv, arcname="results.csv")

print("Saved:")
for p in [final_json, final_weights, zip_path]:
    print("OK " if p.exists() else "MISS ", p)
print(json.dumps(metrics, indent=2))

## Step 8 — Download outputs from Kaggle

After the run finishes, open the right sidebar / Output area and download:

- `/kaggle/working/pyrofinder_results/baseline_yolo11n.json`
- `/kaggle/working/pyrofinder_results/yolo11n_dfire_best.pt`
- `/kaggle/working/pyrofinder_results/pyrofinder_yolo11n_results.zip`

Local PyroFinder repo:

```bash
mkdir -p results
```

Copy:

```text
baseline_yolo11n.json -> results/baseline_yolo11n.json
```

Keep the `.pt` file local in a gitignored folder, for example:

```text
models/yolo11n_dfire_best.pt
```

Then run locally:

```bash
python -m pytest
streamlit run app.py
git status
```